# **Accessing Runtime Inside a Tool**

Modern agent systems are stateful systems where execution is driven by structured data rather than isolated function calls.

**ToolRuntime** provides controlled access to that memory during tool execution.

## **Understanding `ToolRuntime`**

**Tools operate inside a managed runtime with shared memory and context boundaries**

LangChain’s `create_agent` runs on LangGraph’s runtime under the hood. This runtime can be accessed inside the tool using `ToolRuntime`. 

Understand that, when a tool runs, it doesn’t run in isolation. It requires:
- current request info
- agent state
- metadata
- execution context

`ToolRuntime` provides this to a tool.

In [1]:
# Step 1: Init a chat model
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

openai_chat_model = ChatOpenAI(
    api_key=OPENAI_API_KEY, 
    model="gpt-4o-mini", 
    temperature=1
)

In [2]:
# Step 2: Define a Tool which reads the ToolRuntime
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(
    runtime: ToolRuntime
) -> str:
    """Look up user preferences."""
    print("Keys in the ToolRuntime Object:", runtime.__dict__.keys())
    print()
    return f"User preferences: Dark Theme"

In [3]:
# Step 3: Pass the State in create_agent using state_schema arg
from langchain.agents import create_agent

agent = create_agent(
    model=openai_chat_model,
    tools=[get_user_info],
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Hi! My name is Bob. Can you provide my preferences?")],
    }
)

for msg in response["messages"]:
    msg.pretty_print()

Keys in the ToolRuntime Object: dict_keys(['state', 'context', 'config', 'stream_writer', 'tool_call_id', 'store'])

================================ Human Message =================================

Hi! My name is Bob. Can you provide my preferences?
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_QCRm4kj4Pv2SGiYCReznHiMD)
 Call ID: call_QCRm4kj4Pv2SGiYCReznHiMD
  Args:
================================= Tool Message =================================
Name: get_user_info

User preferences: Dark Theme
================================== Ai Message ==================================

Hi Bob! Your preference is set to a Dark Theme. If you need anything else, just let me know!


## **Understanding the `ToolRuntime` Object**

ToolRuntime contains the following:
1. **runtime.state**
    - By default contains "messages" key i.e. `runtime.state["messages"]`
    - You can use this to access the **custom state schema as well** eg: `runtime.state["event"]`
    - State evolves during execution
    - It is mutable memory shared across the agent execution across all the tools
    - You can **persist** the state using a **checkpointer**
    - **Analogy:** Think of this as RAM
2. **runtime.context**
    - Static, external information about the current request
    - You should not update it
    - It comes from outside the tool and remains same across all tools in that request, for eg: user_id, session_id, db_connection_url, etc...
    - Context should NOT be used to store intermediate results
    - **Analogy:** Think of this as RAM
3. **runtime.store**: long-term memory
4. **runtime.config**: configuration like tags, metadata, configurable.thread_id, etc...
5. **runtime.stream_writer**: to share the tool progress
6. **runtime.tool_call_id**: unique id assigned to each tool call

Note that two more keys are added to the ToolRuntime: 'execution_info', 'server_info'

**Note:**
- Context: Immutable and can have a default value
- State: Mutable and can't have a default value

## **Passing `AgentState` to Tool**

In [1]:
# Step 1: Init a chat model
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

openai_chat_model = ChatOpenAI(
    api_key=OPENAI_API_KEY, 
    model="gpt-4o-mini", 
    temperature=1
)

In [13]:
# Step 1: Define a Custom Agent State
from langchain.agents import AgentState

class CustomAgentState(AgentState):  
    user_id: str
    theme_preference: str

In [18]:
from langchain.agents import AgentState
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(
    state: AgentState,
    tool_runtime: ToolRuntime,
) -> str:
    """Look up user preferences."""
    print(f"Agent Runtime Has these keys: {state.keys()}")
    print()
    print(f"Tool Runtime Has these keys: {tool_runtime.state.keys()}")
    print()
    return f"User preferences: Dark Theme"

In [19]:
# Step 3: Pass the State in create_agent using state_schema arg
from langchain.agents import create_agent

agent = create_agent(
    model=openai_chat_model,
    tools=[get_user_info],
    state_schema=CustomAgentState
)

In [20]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Hi! My name is Bob. Can you provide my preferences?")],
        "user_id": "user_123",
        "theme_preference": "dark"
    }, 
    {
        "configurable" : {"thread_id" : "1"}
    }
)

for msg in response["messages"]:
    msg.pretty_print()

Agent Runtime Has these keys: dict_keys(['messages'])

Tool Runtime Has these keys: dict_keys(['messages', 'user_id', 'theme_preference'])

================================ Human Message =================================

Hi! My name is Bob. Can you provide my preferences?
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_mIbOFonrzZG7J6VvqUJ04Meg)
 Call ID: call_mIbOFonrzZG7J6VvqUJ04Meg
  Args:
    state: {'messages': [{'content': 'Hi! My name is Bob. Can you provide my preferences?', 'type': 'human'}]}
================================= Tool Message =================================
Name: get_user_info

User preferences: Dark Theme
================================== Ai Message ==================================

Your preference is set to a Dark Theme. If there's anything else you'd like to know or adjust, feel free to ask!
